In [ ]:
# =============================================================================
# TFM: SISTEMA HÍBRIDO DE DETECCIÓN DE ANOMALÍAS Y MOMENTUM TRADING
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import ta
import os
import warnings

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =============================================================================
# BLOQUE 0 — PARÁMETROS GLOBALES
# =============================================================================

TICKERS_TFM = {
    "spain": ["SAN.MC", "FER.MC", "ACS.MC", "ITX.MC", "REP.MC", "AMS.MC"],
    "usa"  : ["AAPL", "MSFT", "NVDA", "JPM", "BLK", "TSLA", "DPZ", "CAT", "XOM"],
    "etfs" : ["XLK", "SMH", "XLV", "XLI", "XLY", "XLP", "GLD", "SPY", 
              "QQQ", "EWP", "MTUM", "XLE", "XLF", "TLT", "XLU"]
}
ALL_TICKERS = [t for sublist in TICKERS_TFM.values() for t in sublist]

FECHA_INICIO_ANALISIS = "2017-01-01"

# Ventanas de indicadores
W_RSI      = 14
W_MACD_S   = 12
W_MACD_L   = 26
W_MACD_SIG = 9
W_ATR      = 14
W_VOL_REAL = 21       # Corto plazo 
W_VOL_MED  = 63       # ~3 meses 
W_VOL_LARGO = 252     # 1 año bursátil 
W_VOL_ROC  = 5
W_ZSCORE   = 252      # Ventana de estandarización

# Momentum 
W_MOM_6M   = 126      # 6 meses
W_MOM_12M  = 252      # 12 meses
W_MOM_SKIP = 21       # Skip 1 mes 

# Rutas de salida
OUTPUT_DIR_RAW = "data/raw"
OUTPUT_DIR_PROCESSED = "data/processed"
FILE_BRUTO  = os.path.join(OUTPUT_DIR_RAW, "01_dataset_bruto.csv")
FILE_CLEAN  = os.path.join(OUTPUT_DIR_PROCESSED, "02_dataset_final.csv")
FILE_DBSCAN = os.path.join(OUTPUT_DIR_PROCESSED, "03_dataset_dbscan.csv")
FILE_PURGED = os.path.join(OUTPUT_DIR_PROCESSED, "04_dataset_purged_kmeans.csv")

# --- LOOKBACK BUFFER ---
WARMUP_BIZ_DAYS = W_ZSCORE + W_MOM_12M + 35
WARMUP_CAL_DAYS = int(np.ceil(WARMUP_BIZ_DAYS * (365 / 252))) + 30

FECHA_DESCARGA = (
    pd.Timestamp(FECHA_INICIO_ANALISIS) - pd.Timedelta(days=WARMUP_CAL_DAYS)
).strftime("%Y-%m-%d")

print(f"  Análisis desde : {FECHA_INICIO_ANALISIS}")
print(f"  Descarga desde : {FECHA_DESCARGA}  (buffer de {WARMUP_CAL_DAYS} días naturales)")


# =============================================================================
# BLOQUE 1 — DESCARGA DE DATOS
# =============================================================================

def descargar_datos(tickers: list):
    raw = yf.download(
        tickers,
        start=FECHA_DESCARGA,
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    return raw["Close"], raw["High"], raw["Low"], raw["Volume"]

# =============================================================================
# BLOQUE 2 — UNIFICACIÓN DE SERIES TEMPORALES
# =============================================================================

def unificar_series_temporales(close_df, high_df, low_df, volume_df):
    df = pd.concat({
        "Precio_Adj": close_df,
        "High": high_df,
        "Low": low_df,
        "Volumen": volume_df
    }, axis=1)

    df = df.stack(level=1).reset_index()
    df.columns = ["Fecha", "Ticker", "Precio_Adj", "High", "Low", "Volumen"]
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df.sort_values(["Ticker", "Fecha"]).reset_index(drop=True)
    return df


# =============================================================================
# BLOQUE 3 — LIMPIEZA + ELIMINAR FESTIVOS
# =============================================================================

def limpiar_y_eliminar_festivos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Ticker" not in df.columns:
        df = df.reset_index()

    if "Volumen" not in df.columns:
        raise ValueError("Falta columna Volumen en el DataFrame")

    df["Volumen"] = pd.to_numeric(df["Volumen"], errors="coerce")
    filtro_festivos = df["Volumen"].isna() | (df["Volumen"] == 0)
    print(f"  Festivos detectados: {filtro_festivos.sum():,}")

    df = df.loc[~filtro_festivos].copy()

    for col in ["Precio_Adj", "High", "Low"]:
        if col in df.columns:
            df[col] = df.groupby("Ticker")[col].ffill(limit=3)

    df = df.dropna(subset=["Precio_Adj", "High", "Low"])
    return df

# =============================================================================
# BLOQUE 4 — INDICADORES Y CARACTERÍSTICAS
# =============================================================================

def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["Ticker", "Fecha"])

    def add_group(col, func):
        return df.groupby("Ticker")[col].transform(func)

    # ── Retornos
    df["DailyLogReturn"] = add_group("Precio_Adj", lambda s: np.log(s / s.shift(1)))
    df["ALR1W"] = add_group("DailyLogReturn", lambda s: s.rolling(5).sum())
    df["ALR1M"] = add_group("DailyLogReturn", lambda s: s.rolling(21).sum())

    # ── Momentum Institucional (Skip)
    df["ALR12M_SKIP"] = add_group("DailyLogReturn", lambda s: s.rolling(W_MOM_12M).sum() - s.rolling(W_MOM_SKIP).sum())
    df["ALR6M_SKIP"]  = add_group("DailyLogReturn", lambda s: s.rolling(W_MOM_6M).sum() - s.rolling(W_MOM_SKIP).sum())

    # ── Osciladores Técnicos
    df["RSI_14"] = add_group("Precio_Adj", lambda s: ta.momentum.rsi(s, window=W_RSI))
    df["MACD_Hist"] = add_group("Precio_Adj", lambda s: ta.trend.macd_diff(
        s, window_slow=W_MACD_L, window_fast=W_MACD_S, window_sign=W_MACD_SIG
    ))

    # ── Volatilidad
    df["Volatilidad_21d"]  = add_group("DailyLogReturn", lambda s: s.rolling(W_VOL_REAL).std())
    df["Volatilidad_252d"] = add_group("DailyLogReturn", lambda s: s.rolling(W_VOL_LARGO).std())

    df["ATR_14"] = df.groupby("Ticker").apply(
        lambda g: ta.volatility.average_true_range(
            high=g["High"], low=g["Low"], close=g["Precio_Adj"], window=W_ATR
        )
    ).reset_index(level=0, drop=True)

    # ── Volumen
    log_vol = np.log1p(df["Volumen"])
    df["Volumen_ROC"] = df.groupby("Ticker")["Volumen"].transform(
        lambda s: ta.momentum.roc(np.log1p(s), window=W_VOL_ROC)
    )

    vol_media_20 = add_group("Volumen", lambda s: s.rolling(20).mean())
    df["Vol_Relativo"] = df["Volumen"] / vol_media_20.replace(0, np.nan)

    return df


# =============================================================================
# BLOQUE 5 — Z-SCORE ROLLING POR TICKER
# =============================================================================

def estandarizar_zscore_rolling(df: pd.DataFrame) -> pd.DataFrame:
    columnas_Z = [
        "DailyLogReturn", "ALR1W", "ALR1M", "MACD_Hist",
        "ALR12M_SKIP", "ALR6M_SKIP", "Volatilidad_252d",
        "Volatilidad_21d", "ATR_14", "Volumen_ROC",
        "RSI_14", "Vol_Relativo"
    ]

    df = df.sort_values(["Ticker", "Fecha"]).copy()

    for col in columnas_Z:
        if col in df.columns:
            mu = df.groupby("Ticker")[col].transform(
                lambda s: s.rolling(W_ZSCORE, min_periods=W_ZSCORE // 2).mean()
            )
            std = df.groupby("Ticker")[col].transform(
                lambda s: s.rolling(W_ZSCORE, min_periods=W_ZSCORE // 2).std()
            )
            df[f"{col}_Z"] = (df[col] - mu) / std.replace(0, np.nan)

    return df

# =============================================================================
# BLOQUE 6 — RECORTE Y LIMPIEZA FINAL
# =============================================================================

def recortar(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["Fecha"] >= FECHA_INICIO_ANALISIS].copy()
    variables_ml = [c for c in df.columns if c.endswith("_Z")]
    df = df.dropna(subset=variables_ml)
    columnas_finales = ["Fecha", "Ticker"] + variables_ml
    return df[columnas_finales]

# =============================================================================
# BLOQUE 7 — ESPACIO VECTORIAL TOPOLÓGICO (DBSCAN + K-MEANS)
# =============================================================================

FEATURES_DBSCAN = [
    # ── MOMENTUM INSTITUCIONAL
    "ALR12M_SKIP_Z",      
    "ALR6M_SKIP_Z",       
    "ALR1M_Z",            
    # ── TÉCNICO 
    "MACD_Hist_Z",        
    "RSI_14_Z",           
    # ── RIESGO ESTRUCTURAL 
    "Volatilidad_252d_Z", 
    "ATR_14_Z",           
    # ── MICROESTRUCTURA
    "Vol_Relativo_Z",     
]

FEATURES_KMEANS = FEATURES_DBSCAN

def calibrar_eps_kdistance(X: np.ndarray, k: int = 3, percentil: float = 90.0) -> float:
    if len(X) < k + 1:
        return 2.5  
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(X)
    dists, _ = nbrs.kneighbors(X)
    k_dists  = dists[:, -1]
    eps_calculado = float(np.percentile(k_dists, percentil))
    return max(2.5, eps_calculado) 


def aplicar_dbscan_cross_sectional(
    df_clean: pd.DataFrame,
    features: list          = FEATURES_DBSCAN,
    eps_percentil: float    = 90.0,
    min_samples_ratio: float = 0.10,   
    verbose: bool           = True,
) -> pd.DataFrame:

    features_disponibles = [f for f in features if f in df_clean.columns]
    if not features_disponibles:
        raise ValueError("Ninguna de las features DBSCAN está en el DataFrame.")

    resultados = []
    fechas     = df_clean["Fecha"].unique()
    anomalias_por_fecha = []

    for fecha in sorted(fechas):
        corte = df_clean[df_clean["Fecha"] == fecha].copy()
        n     = len(corte)

        if n < 4:  
            corte["DBSCAN_Label"] = 0
            resultados.append(corte)
            continue

        X = corte[features_disponibles].values
        col_means = np.nanmedian(X, axis=0)
        inds_nan  = np.where(np.isnan(X))
        X[inds_nan] = np.take(col_means, inds_nan[1])

        eps          = calibrar_eps_kdistance(X, k=3, percentil=eps_percentil)
        min_s        = max(3, int(np.ceil(n * min_samples_ratio)))

        db           = DBSCAN(eps=eps, min_samples=min_s, metric="euclidean", n_jobs=-1)
        etiquetas    = db.fit_predict(X)

        corte = corte.copy()
        corte["DBSCAN_Label"] = etiquetas
        corte["DBSCAN_eps"]   = round(eps, 4)
        resultados.append(corte)

        n_ruido = (etiquetas == -1).sum()
        if n_ruido > 0:
            anomalias_por_fecha.append({"Fecha": fecha, "N_Anomalias": n_ruido, "N_Total": n})

    df_out = pd.concat(resultados, ignore_index=True)

    if verbose:
        total_filas   = len(df_out)
        total_anomalia = (df_out["DBSCAN_Label"] == -1).sum()
        pct           = 100 * total_anomalia / total_filas
        print(f"\n{'='*60}")
        print(f"  DBSCAN CROSS-SECTIONAL — RESULTADOS")
        print(f"{'='*60}")
        print(f"  Fechas procesadas  : {len(fechas):,}")
        print(f"  Observaciones total: {total_filas:,}")
        print(f"  Anomalías (-1)     : {total_anomalia:,}  ({pct:.2f} %)")
        print(f"{'='*60}")

    return df_out


# =============================================================================
# BLOQUE 8 — DATASET PURGADO PARA K-MEANS
# =============================================================================

def purgar_anomalias_para_kmeans(df_dbscan: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    n_antes = len(df_dbscan)
    n_anomalias = (df_dbscan["DBSCAN_Label"] == -1).sum()

    df_purged = df_dbscan[df_dbscan["DBSCAN_Label"] != -1].copy()

    if verbose:
        n_despues = len(df_purged)
        pct_eliminado = 100 * n_anomalias / n_antes
        print(f"\n{'='*60}")
        print(f"  PURGA PARA K-MEANS")
        print(f"{'='*60}")
        print(f"  Filas antes        : {n_antes:>10,}")
        print(f"  Anomalías borradas : {n_anomalias:>10,}  ({pct_eliminado:.2f} %)")
        print(f"  Filas resultantes  : {n_despues:>10,}")
        print(f"{'='*60}")

    return df_purged

# =============================================================================
# EXPORTACIÓN GLOBAL
# =============================================================================

def exportar_todo(df_bruto, df_clean, df_dbscan, df_purged):
    os.makedirs(OUTPUT_DIR_RAW, exist_ok=True)
    os.makedirs(OUTPUT_DIR_PROCESSED, exist_ok=True)
    
    df_bruto.to_csv(FILE_BRUTO, index=False, sep=";", decimal=",")
    df_clean.to_csv(FILE_CLEAN, index=False, sep=";", decimal=",")
    df_dbscan.to_csv(FILE_DBSCAN, index=False, sep=";", decimal=",")
    df_purged.to_csv(FILE_PURGED, index=False, sep=";", decimal=",")

    print(f"\n{'='*60}")
    print(f"  EXPORTACIÓN FINAL")
    print(f"{'='*60}")
    print(f"  Bruto          → {FILE_BRUTO}   [{len(df_bruto):,} filas]")
    print(f"  Limpio Z-Score → {FILE_CLEAN}   [{len(df_clean):,} filas]")
    print(f"  Con anomalías  → {FILE_DBSCAN}  [{len(df_dbscan):,} filas]")
    print(f"  Purgado K-Means→ {FILE_PURGED}  [{len(df_purged):,} filas]")
    print(f"{'='*60}")

# =============================================================================
# MAIN 
# =============================================================================

if __name__ == "__main__":
    

    print("Descargando datos de mercado...")
    close_df, high_df, low_df, volume_df = descargar_datos(ALL_TICKERS)
    df_bruto = unificar_series_temporales(close_df, high_df, low_df, volume_df)
    
    print("Limpiando festivos...")
    df = limpiar_y_eliminar_festivos(df_bruto)
    
    print("Calculando features...")
    df = calcular_indicadores(df)
    
    print("Estandarizando...")
    df = estandarizar_zscore_rolling(df)
    df_clean = recortar(df)
    
    print("Aplicando DBSCAN...")
    df_dbscan = aplicar_dbscan_cross_sectional(df_clean)
    
    print("Purgando anomalías...")
    df_purged = purgar_anomalias_para_kmeans(df_dbscan)

    print("Exportando archivos...")
    exportar_todo(df_bruto, df_clean, df_dbscan, df_purged)
    print("PROCESO FINALIZADO")

  Análisis desde : 2017-01-01
  Descarga desde : 2014-10-13  (buffer de 811 días naturales)
Descargando datos de mercado...
Limpiando festivos...
  Festivos detectados: 6
Calculando features...
Estandarizando...
Aplicando DBSCAN...

  DBSCAN CROSS-SECTIONAL — RESULTADOS
  Fechas procesadas  : 2,412
  Observaciones total: 70,668
  Anomalías (-1)     : 2,362  (3.34 %)
Purgando anomalías...

  PURGA PARA K-MEANS
  Filas antes        :     70,668
  Anomalías borradas :      2,362  (3.34 %)
  Filas resultantes  :     68,306
Exportando archivos...

  EXPORTACIÓN FINAL
  Bruto          → data/raw\01_dataset_bruto.csv   [87,528 filas]
  Limpio Z-Score → data/processed\02_dataset_final.csv   [70,668 filas]
  Con anomalías  → data/processed\03_dataset_dbscan.csv  [70,668 filas]
  Purgado K-Means→ data/processed\04_dataset_purged_kmeans.csv  [68,306 filas]
PROCESO FINALIZADO
